# C3: ViCrop rel_att (Qwen2.5-VL-3B) trên tập dev

Với mỗi câu dev: trả lời bằng ảnh đầy đủ (baseline) và bằng ảnh crop theo rel_att. Lưu kết quả từng câu vào `results/C3_vicrop/`. Chỉ chạy **dev**; tập test để dành chạy một lần sau khi chốt cấu hình.

Cần upload repo lên `/content/da2` (zip hoặc Drive) gồm `src/`, `data/processed/visr_bench_vi/` và `data/raw/visr_bench_vi/hf/`. Runtime: T4 GPU.

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow numpy
import os, sys
os.chdir('/content/da2'); sys.path.insert(0, '.')
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
import json, time, random, unicodedata
import numpy as np
from pathlib import Path
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from src.crop.vicrop_relatt import rel_att_map, bbox_from_att_map, ATT_LAYER, GENERAL_PROMPT

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

CFG = dict(
    model='Qwen/Qwen2.5-VL-3B-Instruct', dtype='float16', attn='eager',
    max_pixels=1024 * 28 * 28, att_layer=ATT_LAYER, bbox_size=336,
    general_prompt=GENERAL_PROMPT,
    answer_suffix='\nTrả lời ngắn gọn bằng một cụm từ hoặc một câu.',
    max_new_tokens=64, split='dev',
)
processor = AutoProcessor.from_pretrained(CFG['model'], max_pixels=CFG['max_pixels'])
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    CFG['model'], torch_dtype=torch.float16, device_map='auto', attn_implementation='eager').eval()

In [ ]:
def answer(image, question):
    msgs = [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': question + CFG['answer_suffix']}]}]
    text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=CFG['max_new_tokens'], do_sample=False)
    res = processor.batch_decode(out[:, inp['input_ids'].shape[1]:], skip_special_tokens=True)[0]
    return unicodedata.normalize('NFC', res.strip())

dev = [json.loads(l) for l in open('data/processed/visr_bench_vi/qa_vi_haiku_dev.jsonl', encoding='utf-8') if l.strip()]
out_dir = Path('/kaggle/working/C3_vicrop') if os.path.isdir('/kaggle/working') else Path('results/C3_vicrop')
out_dir.mkdir(parents=True, exist_ok=True)
json.dump(CFG, open(out_dir / 'dev_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
fout = open(out_dir / 'dev_raw.jsonl', 'w', encoding='utf-8')

rows = []
for it in dev:
    img = Image.open(f"data/raw/visr_bench_vi/{it['image']}").convert('RGB')
    q = unicodedata.normalize('NFC', it['question'])
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    full = answer(img, q)
    att = rel_att_map(model, processor, img, q + CFG['answer_suffix'], layer=CFG['att_layer'])
    box = bbox_from_att_map(att, img.size, CFG['bbox_size'])
    crop = answer(img.crop(box), q)
    rows.append(dict(qid=it['qid'], document_id=it['document_id'], question=q, gold=it['answer'],
                     pred_full=full, pred_crop=crop, bbox=box, image_size=img.size,
                     sec=round(time.time() - t0, 2), vram_gb=round(torch.cuda.max_memory_allocated() / 2**30, 2)))
    fout.write(json.dumps(rows[-1], ensure_ascii=False) + '\n'); fout.flush()
    print(it['qid'], '| full:', full[:50], '| crop:', crop[:50])

fout.close()
print('\nĐã lưu tại', out_dir)
print('=== dev_raw.jsonl ===')
print(open(out_dir / 'dev_raw.jsonl', encoding='utf-8').read())
